In [48]:
import re
import requests
from bs4 import BeautifulSoup

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.avito.ru/',
}

session = requests.Session()
session.cookies.set('_ga', 'dummy')
session.cookies.set('_ym_uid', 'dummy')

url = input('Ссылка на объявление: ').strip()

try:
    resp = session.get(url, headers=headers, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # ID
    id_span = soup.find('span', {'data-marker': 'item-view/item-id'})
    if id_span:
        text = id_span.get_text(strip=True)
        match = re.search(r'(\d+)', text)
        item_id = match.group(1) if match else None
    else:
        item_id = None

    # Описание
    desc_div = soup.find('div', {'data-marker': 'item-view/item-description'})
    if desc_div:
        description = ' '.join(desc_div.stripped_strings)
    else:
        description = 'Описание не найдено'

    classified = {'itemId': item_id, 'description': description}
    print(classified)

except Exception as e:
    print(f'Ошибка: {e}')

{'itemId': '7950623730', 'description': '🛠️ Kaпитaльный/космeтичeский ремонт любой кoммеpческoй недвижимocти под ключ c гapaнтиeй дo 5 лeт Здравствуйтe! Meня зовут Станиcлав, я руковожу комaндой по рeмонту кoммерчеcкиx помещений. Мы нa pынкe болеe 10 лет и знаем, кaк cделать peмонт, кoтoрый будет пpиноcить деньги, а не пpoблeмы. 📞 Звоните прямo cейчaс — выезд специалиста и смета бесплатно! ❗ Если вы уже сталкивались с ремонтом, вы знаете: это почти всегда стресс, перерасход и срыв сроков. Мы делаем по-другому. Берём на себя всё: — расчёт и планирование — закупку материалов — весь ремонт под ключ — контроль качества — сдачу объекта в срок ✅ Весь транспорт и инструмент свой. ✅ В штате только граждане РФ. ✅ Гарантия на все работы — до 5 лет. 🎁 Напишите мне в чат — я лично проконсультирую вас и подготовлю предварительную смету. Что нас отличает от других: ✅ Мы — мобильные: Можем сделать ремонт в любой деревне. Нам не нужно искать жилье рядом — мы сами решаем все организационные моменты. ✅ 

In [52]:
def clean_description(description):
    # Оставляем только:
    # - буквы: a-z A-Z а-я А-Я ё Ё
    # - цифры: 0-9
    # - пробелы и стандартные знаки препинания
    pattern = r'[^a-zA-Zа-яА-ЯёЁ0-9\s\.,!?;:()\[\]{}\'"-+=/&*#@$%~`—-]'
    cleaned = re.sub(pattern, '', description)
    # Убираем лишние пробелы (множественные заменяем на один)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# Применяем
description_raw = classified['description']
cleaned_description = clean_description(description_raw)

print(cleaned_description)

Kaпитaльный/космeтичeский ремонт любой кoммеpческoй недвижимocти под ключ c гapaнтиeй дo 5 лeт Здравствуйтe! Meня зовут Станиcлав, я руковожу комaндой по рeмонту кoммерчеcкиx помещений. Мы нa pынкe болеe 10 лет и знаем, кaк cделать peмонт, кoтoрый будет пpиноcить деньги, а не пpoблeмы. Звоните прямo cейчaс — выезд специалиста и смета бесплатно! Если вы уже сталкивались с ремонтом, вы знаете: это почти всегда стресс, перерасход и срыв сроков. Мы делаем по-другому. Берём на себя всё: — расчёт и планирование — закупку материалов — весь ремонт под ключ — контроль качества — сдачу объекта в срок Весь транспорт и инструмент свой. В штате только граждане РФ. Гарантия на все работы — до 5 лет. Напишите мне в чат — я лично проконсультирую вас и подготовлю предварительную смету. Что нас отличает от других: Мы — мобильные: Можем сделать ремонт в любой деревне. Нам не нужно искать жилье рядом — мы сами решаем все организационные моменты. Мы — официальные: Только договор. Оплата частями. Мы предост

In [50]:
def normalize_text(text):
    # 1. Приводим к нижнему регистру
    text = text.lower()
    
    # 2. Уменьшаем повторяющиеся буквы (3+ одинаковых подряд → 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Удаляем повторяющиеся знаки препинания
    text = re.sub(r'([.,!?;:"-+=/&*#@$%~`_—-]){2,}', r'\1', text)
    
    return text

# Применяем к cleaned_description
normalized_description = normalize_text(cleaned_description)

print(normalized_description)

kaпитaльный/космeтичeский ремонт любой кoммеpческoй недвижимocти под ключ c гapaнтиeй дo 5 лeт здравствуйтe! meня зовут станиcлав, я руковожу комaндой по рeмонту кoммерчеcкиx помещений. мы нa pынкe болеe 10 лет и знаем, кaк cделать peмонт, кoтoрый будет пpиноcить деньги, а не пpoблeмы. звоните прямo cейчaс — выезд специалиста и смета бесплатно! если вы уже сталкивались с ремонтом, вы знаете: это почти всегда стресс, перерасход и срыв сроков. мы делаем по-другому. берём на себя всё: — расчёт и планирование — закупку материалов — весь ремонт под ключ — контроль качества — сдачу объекта в срок весь транспорт и инструмент свой. в штате только граждане рф. гарантия на все работы — до 5 лет. напишите мне в чат — я лично проконсультирую вас и подготовлю предварительную смету. что нас отличает от других: мы — мобильные: можем сделать ремонт в любой деревне. нам не нужно искать жилье рядом — мы сами решаем все организационные моменты. мы — официальные: только договор. оплата частями. мы предост

In [51]:
def split_into_chunks(text):
    # делим по всем основным разделителям
    chunks = re.split(r'[.!?;,:\(\)\[\]\{\}/\—]|\sи\s|\sили\s', text)

    # чистим пробелы и убираем пустые чанки
    clean_chunks = [c.strip() for c in chunks if c.strip()]

    return clean_chunks

# Применяем к нормализованному тексту
chunks_of_description = split_into_chunks(normalized_description)

print(chunks_of_description)

['kaпитaльный', 'космeтичeский ремонт любой кoммеpческoй недвижимocти под ключ c гapaнтиeй дo 5 лeт здравствуйтe', 'meня зовут станиcлав', 'я руковожу комaндой по рeмонту кoммерчеcкиx помещений', 'мы нa pынкe болеe 10 лет', 'знаем', 'кaк cделать peмонт', 'кoтoрый будет пpиноcить деньги', 'а не пpoблeмы', 'звоните прямo cейчaс', 'выезд специалиста', 'смета бесплатно', 'если вы уже сталкивались с ремонтом', 'вы знаете', 'это почти всегда стресс', 'перерасход', 'срыв сроков', 'мы делаем по-другому', 'берём на себя всё', 'расчёт', 'планирование', 'закупку материалов', 'весь ремонт под ключ', 'контроль качества', 'сдачу объекта в срок весь транспорт', 'инструмент свой', 'в штате только граждане рф', 'гарантия на все работы', 'до 5 лет', 'напишите мне в чат', 'я лично проконсультирую вас', 'подготовлю предварительную смету', 'что нас отличает от других', 'мы', 'мобильные', 'можем сделать ремонт в любой деревне', 'нам не нужно искать жилье рядом', 'мы сами решаем все организационные моменты',